# 05 — סימולציית שיבוש ברמת הרצף

המחברת מחשבת **מה היה נמחק מרצף הייחוס** אילו שני חיתוכי SpCas9 אידאליים התרחשו. זו אינה סימולציה של תא, של מערכת חיסון או של ריפוי HSV.

המטרה היא להבין את שכבת הקואורדינטות לפני שעוברים לכלי off-target ולנתוני ניסוי.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from viral_safe_target import (
    annotate_candidates,
    rank_candidate_pairs,
    read_fasta,
    read_gff3,
    scan_spcas9_candidates,
)


## 1. טוענים את הדמו הסינתטי

בנתונים אמיתיים נחליף את הקבצים ב-alignment של זני וירוס וב-GFF3 המתאים ל-reference.

In [ ]:
records = read_fasta(ROOT / 'data/demo/virus_aligned.fasta')
features = read_gff3(ROOT / 'data/demo/reference.gff3')

candidates = scan_spcas9_candidates(
    records,
    reference_id='HSV2_demo_ref',
    min_site_coverage=0.0,
)
candidates = annotate_candidates(
    candidates,
    features,
    seqid='HSV2_demo_ref',
)
candidates[['candidate_id', 'strand', 'reference_start_1based',
            'reference_end_1based', 'virus_site_coverage', 'gene_name']]


## 2. מדמים זוגות חיתוך

לכל זוג המערכת מחשבת את שתי נקודות החיתוך הקנוניות, את טווח המחיקה, את החפיפה ל-GFF ואת אחוז הזנים שבהם **שני** האתרים קיימים בדיוק.

In [ ]:
pairs = rank_candidate_pairs(
    candidates,
    features=features,
    aligned_records=records,
    reference_id='HSV2_demo_ref',
    same_feature_only=True,
    min_distance_bp=1,
    max_distance_bp=10_000,
)

pairs.head(10)


## 3. מפרשים תוצאה אחת

- `deletion_length_bp` הוא גודל המחיקה המתמטית בין נקודות החיתוך.
- `exact_pair_coverage` הוא שיעור הזנים שבהם שני אתרי המטרה קיימים ללא שינוי.
- `overlapping_features` מתאר אילו annotations נחתכים.
- `sequence_disruption_score` הוא דירוג שקוף ולא מאומת; הוא **אינו** סיכוי לריפוי או להשבתת הווירוס.

In [ ]:
if pairs.empty:
    print('No eligible pairs were found under the current filters.')
else:
    top = pairs.iloc[0]
    display(top.to_frame('value'))


## 4. איך יודעים מה קרה באמת?

השלב הבא אינו עוד סימולציית Python:

1. סריקת off-target מול גנום המארח באמצעות כלי ייעודי.
2. אם קיימים נתוני sequencing מניסוי עריכה, ניתוחם ב-CRISPResso2 או כלי מקביל.
3. מדידת אפקט וירולוגי במעבדה: כמות DNA ויראלי, יצירת וירוסים מדבקים, reactivation ובטיחות תאית.

ללא נתונים כאלה אפשר לומר רק: *זהו מועמד חישובי בעל תכונות מסוימות*.